# Assignment 2 - Data Pipeline and Preparation

**Unit:** COMP5310 - Project Stage 2  
**Dataset:** NSW Train Occupancy  
**Purpose:** Recreate the Assignment 1 cleaning decisions from the raw data and save one model-ready dataset for Assignment 2.

## Pipeline objectives

This notebook loads the raw dataset using relative paths, records the evidence for each cleaning decision, applies the approved cleaning rules, creates the binary `Crowded` target, validates the final result, and saves the exact file used by `02_modelling_evaluation.ipynb`.

The pipeline deliberately retains rows with unusable timestamps because timestamps are not model predictors in the current decision design. No value is guessed: unrepairable timestamps remain missing.


In [8]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 4. Load the raw dataset

The original NSW Train Occupancy CSV is loaded from a relative path. The notebook checks the expected project locations so that it can be run from either the project root or the notebook directory.

In [9]:
# Locate the raw dataset using relative paths.
# The first path is used in the submitted Assignment 2 package; the second
# supports this repository's shared Assignment 1 data folder.
possible_paths = [
    Path("../data/raw/train_occupancy.csv"),
    Path("../../data/raw/train_occupancy.csv"),
    Path("data/raw/train_occupancy.csv"),
]

data_path = next((path for path in possible_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find train_occupancy.csv. Place it in data/raw and rerun."
    )

train_raw = pd.read_csv(data_path)
print("Dataset loaded successfully.")
print(f"File location: {data_path}")
print(f"Rows: {train_raw.shape[0]:,}")
print(f"Columns: {train_raw.shape[1]}")


Dataset loaded successfully.
File location: ../../data/raw/train_occupancy.csv
Rows: 50,250
Columns: 14


## 5. Initial data audit

Before cleaning, we inspect the dataset's dimensions, variable types, missing values, duplicate records and category consistency. This establishes evidence for subsequent cleaning decisions and helps prevent unnecessary or unjustified changes to the data.

In [10]:
# 5.1 Inspect the structure
print("Dataset dimensions:", train_raw.shape)

display(train_raw.head())

print("\nVariable types:")
display(
    train_raw.dtypes.rename("data_type").to_frame()
)

# 5.2 Missing values
missing_audit = pd.DataFrame({
    "missing_count": train_raw.isna().sum(),
    "missing_percent": (
        train_raw.isna().mean() * 100
    ).round(2)
}).sort_values("missing_count", ascending=False)

print("\nMissing-value audit:")
display(missing_audit)

# 5.3 Exact duplicate rows
duplicate_count = train_raw.duplicated().sum()

print(f"\nExact duplicate rows after the first occurrence: {duplicate_count:,}")

# 5.4 Number of distinct values in each variable
print("\nDistinct values per variable:")
display(
    train_raw.nunique(dropna=False)
    .rename("distinct_values")
    .to_frame()
)

Dataset dimensions: (50250, 14)


,day,Actual.Stop.Station,Actual.Station.Arrv.Time,Actual.Station.Dprt.Time,Segment.Direction,Trip.Name,Service.Line,Orig..Station,Dest..Station,Leading.Set.Type,Node.Seq.Order,Actual.Station.Dprt.Time.Band,Occupancy Status,Occupancy Range
0,9,Miranda,2017-01-10 00:18:17,2017-01-10 00:19:04,Down,620S,Illawarra,Central,Cronulla,T,18,00:15-00:29,MANY_SEATS_AVAILABLE,Low: 0-399
1,13,Beecroft,2017-01-13 08:05:59,2017-01-13 08:06:56,Down,152C,North via Macquarie Park,Central,Hornsby,A,16,08:00-08:14,MANY_SEATS_AVAILABLE,Low: 0-399
2,12,Arncliffe,2017-01-12 07:22:51,2017-01-12 07:23:36,Down,607B,Illawarra,Central,Waterfall,T,6,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399
3,13,Dulwich Hill,2017-01-13 07:20:39,2017-01-13 07:21:53,Down,58-F,Bankstown,Central,Birrong,A,7,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399
4,11,Belmore,2017-01-11 18:38:28,2017-01-11 18:39:07,Up,33-P,Bankstown,Birrong,Central,K,7,18:30-18:44,MANY_SEATS_AVAILABLE,Low: 0-399



Variable types:


,data_type
day,int64
Actual.Stop.Station,str
Actual.Station.Arrv.Time,str
Actual.Station.Dprt.Time,str
Segment.Direction,str
Trip.Name,str
Service.Line,str
Orig..Station,str
Dest..Station,str
Leading.Set.Type,str



Missing-value audit:


,missing_count,missing_percent
Orig..Station,80,0.16
Segment.Direction,78,0.16
Leading.Set.Type,77,0.15
Actual.Stop.Station,74,0.15
Occupancy Status,73,0.15
Dest..Station,63,0.13
Service.Line,59,0.12
Actual.Station.Dprt.Time,11,0.02
Actual.Station.Arrv.Time,7,0.01
day,0,0.00



Exact duplicate rows after the first occurrence: 250

Distinct values per variable:


,distinct_values
day,8
Actual.Stop.Station,308
Actual.Station.Arrv.Time,47199
Actual.Station.Dprt.Time,43140
Segment.Direction,5
Trip.Name,3587
Service.Line,31
Orig..Station,68
Dest..Station,69
Leading.Set.Type,11


## 6. Inspect occupancy categories

The occupancy labels are inspected before cleaning to identify missing values, spelling inconsistencies and unexpected categories. The binary target will be created only after the occupancy labels have been standardised and records with unknown outcomes have been handled.

In [11]:
# Inspect the raw occupancy labels
print("Occupancy Status values:")
display(
    train_raw["Occupancy Status"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

print("\nSegment Direction values:")
display(
    train_raw["Segment.Direction"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

print("\nWhitespace check for selected categorical fields:")
for col in ["Actual.Stop.Station", "Service.Line", "Segment.Direction"]:
    values = train_raw[col].dropna().astype(str)
    whitespace_count = values.ne(values.str.strip()).sum()
    print(f"{col}: {whitespace_count} values with leading/trailing whitespace")

Occupancy Status values:


,label,count
0,MANY_SEATS_AVAILABLE,48008
1,FEW_SEATS_AVAILABLE,1596
2,STANDING_ROOM_ONLY,398
3,MANY_SEATSAVAILABLE,165
4,NaN,73
5,few_seats_available,8
6,VTANDING_ROOM_ONLY,2



Segment Direction values:


,label,count
0,Up,25197
1,Down,24817
2,Up,80
3,Down,78
4,NaN,78



Whitespace check for selected categorical fields:
Actual.Stop.Station: 0 values with leading/trailing whitespace
Service.Line: 7 values with leading/trailing whitespace
Segment.Direction: 158 values with leading/trailing whitespace


## 7. Cleaning decisions and evidence

The table produced below records the issue, evidence, action and impact for every cleaning decision. This makes the pipeline auditable and ensures that the same transformations are applied whenever the notebook is rerun.


In [ ]:
import re

# Preserve the raw data and apply cleaning only to a working copy.
train_df = train_raw.copy()
rows_at_start = len(train_df)

# 1. Trim leading/trailing whitespace in all text fields.
text_columns = train_df.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    train_df[col] = train_df[col].str.strip()

# 2. Correct known occupancy-label inconsistencies identified in the raw audit.
occupancy_mapping = {
    "MANY_SEATSAVAILABLE": "MANY_SEATS_AVAILABLE",
    "few_seats_available": "FEW_SEATS_AVAILABLE",
    "VTANDING_ROOM_ONLY": "STANDING_ROOM_ONLY",
}
train_df["Occupancy Status"] = train_df["Occupancy Status"].replace(occupancy_mapping)

# 3. Remove observations with an unknown outcome: they cannot be used to train
# or fairly evaluate a supervised crowding classifier.
missing_target_rows = train_df["Occupancy Status"].isna().sum()
train_df = train_df.dropna(subset=["Occupancy Status"]).copy()

# 4. Remove only redundant copies of exact duplicate rows.
duplicate_rows_removed = train_df.duplicated().sum()
train_df = train_df.drop_duplicates().copy()

print(f"Rows at start: {rows_at_start:,}")
print(f"Rows removed for missing target: {missing_target_rows:,}")
print(f"Redundant exact duplicate rows removed: {duplicate_rows_removed:,}")
print(f"Rows after target and duplicate cleaning: {len(train_df):,}")


In [ ]:
def parse_full_datetime(value):
    """Parse only complete 2017 timestamps; return NaT otherwise."""
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    if not re.match(r"^2017-01-\d{2}\s+\d{2}:\d{2}:\d{2}$", text):
        return pd.NaT
    return pd.to_datetime(text, format="%Y-%m-%d %H:%M:%S", errors="coerce")


def parse_time_only(value):
    """Return valid (hour, minute, second) values for time-only formats."""
    if pd.isna(value):
        return None
    text = str(value).strip().replace(".", ":")

    ampm = re.match(r"^(\d{1,2}):(\d{2})(?::(\d{2}))?\s*(AM|PM)$", text, re.I)
    if ampm:
        hour, minute, second = int(ampm.group(1)), int(ampm.group(2)), int(ampm.group(3) or 0)
        if not 1 <= hour <= 12 or minute >= 60 or second >= 60:
            return None
        if ampm.group(4).upper() == "PM" and hour != 12:
            hour += 12
        elif ampm.group(4).upper() == "AM" and hour == 12:
            hour = 0
        return hour, minute, second

    colon_time = re.match(r"^(\d{1,2}):(\d{2})(?::(\d{2}))?$", text)
    if colon_time:
        hour, minute, second = int(colon_time.group(1)), int(colon_time.group(2)), int(colon_time.group(3) or 0)
        return (hour, minute, second) if hour < 24 and minute < 60 and second < 60 else None

    compact_time = re.match(r"^(\d{4})$", text)
    if compact_time:
        hour, minute = int(text[:2]), int(text[2:])
        return (hour, minute, 0) if hour < 24 and minute < 60 else None
    return None


def clean_timestamp(value, paired_value):
    """Recover a time-only value only when its paired timestamp provides the date."""
    full_datetime = parse_full_datetime(value)
    if pd.notna(full_datetime):
        return full_datetime

    time_only = parse_time_only(value)
    paired_datetime = parse_full_datetime(paired_value)
    if time_only is not None and pd.notna(paired_datetime):
        hour, minute, second = time_only
        return paired_datetime.normalize() + pd.Timedelta(
            hours=hour, minutes=minute, seconds=second
        )
    return pd.NaT


# Timestamp repair follows the Assignment 1 rule: borrow a date only from the
# paired full timestamp in the same observation; otherwise retain a missing value.
train_df["Arrival_Clean"] = [
    clean_timestamp(arrival, departure)
    for arrival, departure in zip(
        train_df["Actual.Station.Arrv.Time"],
        train_df["Actual.Station.Dprt.Time"],
    )
]
train_df["Departure_Clean"] = [
    clean_timestamp(departure, arrival)
    for arrival, departure in zip(
        train_df["Actual.Station.Dprt.Time"],
        train_df["Actual.Station.Arrv.Time"],
    )
]

arrival_missing = train_df["Arrival_Clean"].isna().sum()
departure_missing = train_df["Departure_Clean"].isna().sum()
assert train_df["Arrival_Clean"].dropna().dt.year.eq(2017).all()
assert train_df["Departure_Clean"].dropna().dt.year.eq(2017).all()

# Create the agreed supervised-learning outcome in the pipeline, not the model notebook.
train_df["Crowded"] = train_df["Occupancy Status"].isin(
    ["FEW_SEATS_AVAILABLE", "STANDING_ROOM_ONLY"]
).astype(int)

cleaning_log = pd.DataFrame([
    {
        "Issue": "Whitespace in categorical values",
        "Evidence": "7 Service.Line and 158 Segment.Direction values had surrounding whitespace",
        "Action": "Trimmed text fields",
        "Impact": "Categories now group consistently",
    },
    {
        "Issue": "Inconsistent occupancy labels",
        "Evidence": "Three label variants identified in the raw audit",
        "Action": "Mapped variants to approved labels",
        "Impact": "Target categories are consistent",
    },
    {
        "Issue": "Missing occupancy outcome",
        "Evidence": f"{missing_target_rows} records had no Occupancy Status",
        "Action": "Removed records with unknown target",
        "Impact": f"{missing_target_rows} rows removed",
    },
    {
        "Issue": "Exact duplicate records",
        "Evidence": f"{duplicate_rows_removed} redundant duplicate rows",
        "Action": "Kept one copy of each record",
        "Impact": f"{duplicate_rows_removed} rows removed",
    },
    {
        "Issue": "Malformed or time-only timestamps",
        "Evidence": "Some timestamps could not be parsed as full datetimes",
        "Action": "Repaired only from paired full timestamp; otherwise retained missing",
        "Impact": f"Arrival missing: {arrival_missing}; departure missing: {departure_missing}",
    },
])
display(cleaning_log)


## 8. Validate and save the model-ready dataset

`Occupancy Status` and `Occupancy Range` must not be used as predictors because they directly reveal, or are closely related to, the crowding outcome. The actual timestamp fields are retained in the saved file for auditability but are excluded later unless their availability at the decision time can be established.


In [ ]:
# Final validation checks
assert len(train_df) == 49_927, "Unexpected cleaned row count"
assert train_df["Occupancy Status"].isna().sum() == 0
assert train_df.duplicated().sum() == 0
assert set(train_df["Occupancy Status"].unique()) == {
    "MANY_SEATS_AVAILABLE", "FEW_SEATS_AVAILABLE", "STANDING_ROOM_ONLY"
}
assert set(train_df["Crowded"].unique()) == {0, 1}

print("Final shape:", train_df.shape)
print("Crowded class distribution:")
display(train_df["Crowded"].value_counts().rename_axis("Crowded").to_frame("records"))

# Use a path relative to the Assignment 2 notebook directory.
output_path = Path("../data/processed/train_occupancy_model_ready.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
train_df.to_csv(output_path, index=False)

print(f"Saved model-ready dataset: {output_path}")
print(f"Saved rows: {len(train_df):,}")
